# AEZ-26 Phase 2C — Flash-Drought Detection

This notebook implements the **locked Phase 2B methodology**.

Primary event variable: `SM_0_100cm_mm`  
Temporal unit: climatological pentads  
Primary criteria: >40th percentile → <=20th percentile, >=5 percentile points/pentad, <=4 transitions, >=3-pentad duration, recovery at >20th percentile.

The notebook does **not** modify the Phase 2A raw daily backbone.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE2_DIR = BASE_DIR / 'Phase2'
PHASE2_DIR.mkdir(exist_ok=True)

INPUT = PHASE2_DIR / 'AEZ26_Daily_Hydroclimate_Backbone_2000_2025.csv'

if not INPUT.exists():
    raise FileNotFoundError(f'Backbone not found: {INPUT}')

daily = pd.read_csv(INPUT, parse_dates=['date']).sort_values('date').reset_index(drop=True)

expected_dates = pd.date_range('2000-01-01', '2025-12-31', freq='D')

assert len(daily) == 9497
assert daily['date'].is_unique
assert pd.DatetimeIndex(daily['date']).equals(expected_dates)
assert daily.isna().sum().sum() == 0

print('Phase 2A backbone loaded:', daily.shape)
print('Input gate: PASS')


## 1. Build seasonally aligned climatological pentads

Leap-day handling is explicit: Feb 29 is retained inside the same climatological pentad as Feb 28. This avoids shifting every post-February seasonal pentad in leap years.


In [ ]:
is_leap = daily['date'].dt.is_leap_year
doy = daily['date'].dt.dayofyear

after_feb29 = is_leap & (daily['date'].dt.month > 2)
clim_doy = doy - after_feb29.astype(int)

feb29 = (daily['date'].dt.month == 2) & (daily['date'].dt.day == 29)
clim_doy = clim_doy.where(~feb29, 59)

daily['clim_doy'] = clim_doy.astype(int)
daily['year'] = daily['date'].dt.year.astype(int)
daily['pentad_of_year'] = (((daily['clim_doy'] - 1) // 5) + 1).astype(int)

assert daily['pentad_of_year'].between(1, 73).all()
assert (daily.groupby('year')['pentad_of_year'].nunique() == 73).all()

print('73 climatological pentads/year: PASS')


In [ ]:
pentad = daily.groupby(['year', 'pentad_of_year'], as_index=False).agg(
    pentad_start=('date', 'min'),
    pentad_end=('date', 'max'),
    n_days=('date', 'count'),

    precipitation_sum_mm=('precipitation_mm_day', 'sum'),
    precipitation_mean_mm_day=('precipitation_mm_day', 'mean'),

    t2m_mean_C=('t2m_mean_C', 'mean'),
    t2m_max_C=('t2m_max_C', 'max'),
    t2m_min_C=('t2m_min_C', 'min'),
    dewpoint_mean_C=('dewpoint_mean_C', 'mean'),
    relative_humidity_mean_pct=('relative_humidity_mean_pct', 'mean'),
    VPD_mean_kPa=('VPD_mean_kPa', 'mean'),
    VPD_max_kPa=('VPD_max_kPa', 'max'),
    wind_speed_mean_m_s=('wind_speed_mean_m_s', 'mean'),

    solar_radiation_sum_MJ_m2=('solar_radiation_MJ_m2_day', 'sum'),
    solar_radiation_mean_MJ_m2_day=('solar_radiation_MJ_m2_day', 'mean'),
    net_solar_radiation_sum_MJ_m2=('net_solar_radiation_MJ_m2_day', 'sum'),
    net_thermal_radiation_sum_MJ_m2=('net_thermal_radiation_MJ_m2_day', 'sum'),
    net_radiation_sum_MJ_m2=('net_radiation_MJ_m2_day', 'sum'),
    net_radiation_mean_MJ_m2_day=('net_radiation_MJ_m2_day', 'mean'),

    total_evaporation_sum_mm=('total_evaporation_mm_day', 'sum'),
    total_evaporation_mean_mm_day=('total_evaporation_mm_day', 'mean'),
    ERA5_potential_evaporation_sum_mm=('ERA5_potential_evaporation_mm_day', 'sum'),
    ERA5_potential_evaporation_mean_mm_day=('ERA5_potential_evaporation_mm_day', 'mean'),

    SM_0_10cm_mean_mm=('SM_0_10cm_mm', 'mean'),
    SM_10_40cm_mean_mm=('SM_10_40cm_mm', 'mean'),
    SM_40_100cm_mean_mm=('SM_40_100cm_mm', 'mean'),
    SM_100_200cm_mean_mm=('SM_100_200cm_mm', 'mean'),
    SM_0_40cm_mean_mm=('SM_0_40cm_mm', 'mean'),
    SM_0_100cm_mean_mm=('SM_0_100cm_mm', 'mean'),
    SM_0_200cm_mean_mm=('SM_0_200cm_mm', 'mean'),
    RootMoist_mean_mm=('RootMoist_mm', 'mean'),

    surface_runoff_sum_mm=('surface_runoff_mm_day', 'sum'),
    surface_runoff_mean_mm_day=('surface_runoff_mm_day', 'mean'),
    baseflow_sum_mm=('baseflow_subsurface_mm_day', 'sum'),
    baseflow_mean_mm_day=('baseflow_subsurface_mm_day', 'mean'),
    total_runoff_sum_mm=('total_runoff_mm_day', 'sum'),
    total_runoff_mean_mm_day=('total_runoff_mm_day', 'mean'),
    GLDAS_ET_sum_mm=('GLDAS_ET_mm_day', 'sum'),
    GLDAS_ET_mean_mm_day=('GLDAS_ET_mm_day', 'mean'),

    CHIRPS_valid_fraction_min=('CHIRPS_valid_pixel_fraction', 'min'),
    ERA5_complete_fraction=('ERA5_complete_day_flag', 'mean'),
    GLDAS_state_complete_fraction=('GLDAS_state_complete_flag', 'mean'),
    GLDAS_flux_complete_fraction=('GLDAS_flux_complete_flag', 'mean'),
    GLDAS_complete_fraction=('GLDAS_complete_day_flag', 'mean'),
)

pentad = pentad.sort_values('pentad_start').reset_index(drop=True)
pentad.insert(0, 'pentad_id', np.arange(1, len(pentad) + 1))
pentad['pentad_label'] = (
    pentad['year'].astype(str)
    + '_P'
    + pentad['pentad_of_year'].astype(str).str.zfill(2)
)
pentad['pentad_mid_date'] = (
    pentad['pentad_start']
    + (pentad['pentad_end'] - pentad['pentad_start']) / 2
)
pentad['leap_day_included_flag'] = (pentad['n_days'] == 6).astype(int)

assert len(pentad) == 1898
assert int((pentad['n_days'] == 6).sum()) == 7

print('Pentad table:', pentad.shape)
print(pentad['n_days'].value_counts().sort_index())


## 2. Soil-moisture percentiles

Primary percentile = empirical rank among the same climatological pentad across 2000–2025.

Two pre-specified sensitivity percentiles are retained:
- `RootMoist_pct`
- `SM_0_100_pct_window5` using a ±2-pentad seasonal reference.


In [ ]:
pentad['SM_0_100_pct'] = (
    pentad.groupby('pentad_of_year')['SM_0_100cm_mean_mm']
    .rank(method='average', pct=True)
    .mul(100.0)
)

pentad['RootMoist_pct'] = (
    pentad.groupby('pentad_of_year')['RootMoist_mean_mm']
    .rank(method='average', pct=True)
    .mul(100.0)
)

def empirical_rank_percentile(value, reference):
    ref = np.asarray(reference, dtype=float)
    ref = ref[np.isfinite(ref)]
    less = np.sum(ref < value)
    equal = np.sum(ref == value)
    average_rank = less + (equal + 1.0) / 2.0
    return 100.0 * average_rank / len(ref)

window_pct = []
for row in pentad.itertuples(index=False):
    p = int(row.pentad_of_year)
    seasonal_window = [((p - 1 + offset) % 73) + 1 for offset in range(-2, 3)]
    reference = pentad.loc[
        pentad['pentad_of_year'].isin(seasonal_window),
        'SM_0_100cm_mean_mm'
    ]
    window_pct.append(
        empirical_rank_percentile(row.SM_0_100cm_mean_mm, reference)
    )

pentad['SM_0_100_pct_window5'] = window_pct
pentad['SM_0_100_pct_change_1pentad'] = pentad['SM_0_100_pct'].diff()
pentad['SM_0_100_drying_rate_pp_1pentad'] = -pentad['SM_0_100_pct_change_1pentad']

print('Primary percentile missing:', pentad['SM_0_100_pct'].isna().sum())
print('RootMoist percentile missing:', pentad['RootMoist_pct'].isna().sum())
print('Window-5 percentile missing:', pentad['SM_0_100_pct_window5'].isna().sum())


## 3. Locked flash-drought event detector

The >40th-percentile pentad is a **pre-onset anchor**. The event begins in the following pentad. Recovery is the first pentad above the 20th percentile after the <=20th-percentile crossing.


In [ ]:
def detect_flash_droughts(
    pentad_df,
    pct_col,
    catalogue_name,
    start_threshold=40.0,
    drought_threshold=20.0,
    min_decline_rate=5.0,
    max_intensification_transitions=4,
    min_event_duration_pentads=3
):
    x = pentad_df.sort_values('pentad_start').reset_index(drop=True).copy()
    pct = x[pct_col].to_numpy(dtype=float)

    events = []
    i = 0
    event_number = 1
    n = len(x)

    while i < n - 1:

        if not np.isfinite(pct[i]) or pct[i] <= start_threshold:
            i += 1
            continue

        accepted = False

        for j in range(
            i + 1,
            min(n, i + max_intensification_transitions + 1)
        ):
            if not np.isfinite(pct[j]):
                break

            if j > i + 1 and np.any(pct[i + 1:j] > start_threshold):
                break

            if pct[j] <= drought_threshold:
                decline_rate = (pct[i] - pct[j]) / (j - i)

                if decline_rate < min_decline_rate:
                    continue

                onset_idx = i + 1

                recovery_idx = j + 1
                while (
                    recovery_idx < n
                    and np.isfinite(pct[recovery_idx])
                    and pct[recovery_idx] <= drought_threshold
                ):
                    recovery_idx += 1

                right_censored = (
                    recovery_idx >= n
                    or (
                        recovery_idx < n
                        and not np.isfinite(pct[recovery_idx])
                    )
                )

                event_end_idx = recovery_idx - 1
                duration = event_end_idx - onset_idx + 1

                if duration < min_event_duration_pentads:
                    continue

                event_slice = x.loc[onset_idx:event_end_idx].copy()
                severity = float(
                    np.maximum(
                        0.0,
                        start_threshold - event_slice[pct_col].to_numpy(dtype=float)
                    ).sum()
                )

                ant_start = max(0, onset_idx - 4)
                antecedent = x.loc[ant_start:onset_idx - 1].copy()

                recovery_date = (
                    pd.NaT
                    if right_censored
                    else x.loc[recovery_idx, 'pentad_mid_date']
                )
                recovery_label = (
                    None
                    if right_censored
                    else x.loc[recovery_idx, 'pentad_label']
                )

                events.append({
                    'catalogue': catalogue_name,
                    'event_id': f'{catalogue_name}_FD_{event_number:03d}',

                    'anchor_pentad': x.loc[i, 'pentad_label'],
                    'anchor_mid_date': x.loc[i, 'pentad_mid_date'],
                    'anchor_percentile': pct[i],

                    'onset_pentad': x.loc[onset_idx, 'pentad_label'],
                    'onset_start_date': x.loc[onset_idx, 'pentad_start'],
                    'onset_mid_date': x.loc[onset_idx, 'pentad_mid_date'],

                    'threshold_crossing_pentad': x.loc[j, 'pentad_label'],
                    'threshold_crossing_mid_date': x.loc[j, 'pentad_mid_date'],
                    'threshold_crossing_percentile': pct[j],

                    'event_end_pentad': x.loc[event_end_idx, 'pentad_label'],
                    'event_end_date': x.loc[event_end_idx, 'pentad_end'],

                    'recovery_pentad': recovery_label,
                    'termination_mid_date': recovery_date,
                    'right_censored_flag': int(right_censored),

                    'intensification_duration_pentads': int(j - i),
                    'total_duration_pentads': int(duration),
                    'total_duration_actual_days': int(event_slice['n_days'].sum()),

                    'mean_intensification_rate_pp_per_pentad': float(decline_rate),
                    'minimum_soil_moisture_percentile': float(event_slice[pct_col].min()),
                    'severity_40pct_deficit_pp_pentads': severity,

                    'onset_year': int(x.loc[onset_idx, 'year']),
                    'onset_month': int(x.loc[onset_idx, 'pentad_mid_date'].month),

                    'antecedent_4pentad_precipitation_mm': float(
                        antecedent['precipitation_sum_mm'].sum()
                    ),
                    'antecedent_4pentad_VPD_mean_kPa': float(
                        np.average(
                            antecedent['VPD_mean_kPa'],
                            weights=antecedent['n_days']
                        )
                    ) if len(antecedent) else np.nan,

                    'event_precipitation_mm': float(
                        event_slice['precipitation_sum_mm'].sum()
                    ),
                    'event_VPD_mean_kPa': float(
                        np.average(
                            event_slice['VPD_mean_kPa'],
                            weights=event_slice['n_days']
                        )
                    ),
                    'event_VPD_max_kPa': float(event_slice['VPD_max_kPa'].max()),
                    'event_t2m_mean_C': float(
                        np.average(
                            event_slice['t2m_mean_C'],
                            weights=event_slice['n_days']
                        )
                    ),
                    'event_t2m_max_C': float(event_slice['t2m_max_C'].max()),
                    'event_ERA5_potential_evaporation_mm': float(
                        event_slice['ERA5_potential_evaporation_sum_mm'].sum()
                    ),
                    'event_GLDAS_ET_mm': float(
                        event_slice['GLDAS_ET_sum_mm'].sum()
                    ),
                })

                event_number += 1
                accepted = True
                i = recovery_idx if not right_censored else n
                break

        if not accepted:
            i += 1

    return pd.DataFrame(events)

events_primary = detect_flash_droughts(
    pentad, 'SM_0_100_pct', 'PRIMARY_SM0_100'
)

events_root = detect_flash_droughts(
    pentad, 'RootMoist_pct', 'SENS_ROOTMOIST'
)

events_window = detect_flash_droughts(
    pentad, 'SM_0_100_pct_window5', 'SENS_WINDOW5'
)

print('Primary events:', len(events_primary))
print('RootMoist sensitivity events:', len(events_root))
print('Window-5 sensitivity events:', len(events_window))


In [ ]:
# Add primary event-phase flags to the pentad table.
pentad['primary_flash_drought_event_flag'] = 0
pentad['primary_flash_drought_event_id'] = ''
pentad['primary_flash_drought_intensification_flag'] = 0
pentad['primary_flash_drought_drought_phase_flag'] = 0

label_to_idx = {
    label: idx for idx, label in enumerate(pentad['pentad_label'])
}

for ev in events_primary.itertuples(index=False):
    onset_idx = label_to_idx[ev.onset_pentad]
    cross_idx = label_to_idx[ev.threshold_crossing_pentad]
    end_idx = label_to_idx[ev.event_end_pentad]

    pentad.loc[onset_idx:end_idx, 'primary_flash_drought_event_flag'] = 1
    pentad.loc[onset_idx:end_idx, 'primary_flash_drought_event_id'] = ev.event_id
    pentad.loc[onset_idx:cross_idx, 'primary_flash_drought_intensification_flag'] = 1
    pentad.loc[cross_idx:end_idx, 'primary_flash_drought_drought_phase_flag'] = 1

print('Primary event pentads:', pentad['primary_flash_drought_event_flag'].sum())


## 4. Save Phase 2C outputs

The primary catalogue and both sensitivity catalogues are saved separately so they cannot be accidentally conflated.


In [ ]:
PHASE2C_DIR = PHASE2_DIR / 'Phase2C'
PHASE2C_DIR.mkdir(exist_ok=True)

PENTAD_OUT = PHASE2C_DIR / 'AEZ26_Pentad_Hydroclimate_Indicators_2000_2025.csv'
PRIMARY_OUT = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Primary_2000_2025.csv'
ROOT_OUT = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Sensitivity_RootMoist_2000_2025.csv'
WINDOW_OUT = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Sensitivity_Window5_2000_2025.csv'
QC_OUT = PHASE2C_DIR / 'AEZ26_Phase2C_Flash_Drought_Detection_QC.csv'

pentad.to_csv(PENTAD_OUT, index=False, date_format='%Y-%m-%d')
events_primary.to_csv(PRIMARY_OUT, index=False, date_format='%Y-%m-%d')
events_root.to_csv(ROOT_OUT, index=False, date_format='%Y-%m-%d')
events_window.to_csv(WINDOW_OUT, index=False, date_format='%Y-%m-%d')

qc_rows = [
    ['Daily input rows', len(daily), 'PASS'],
    ['Pentad rows', len(pentad), 'PASS'],
    ['5-day pentads', int((pentad['n_days'] == 5).sum()), 'PASS'],
    ['6-day leap pentads', int((pentad['n_days'] == 6).sum()), 'PASS'],
    ['Primary percentile missing', int(pentad['SM_0_100_pct'].isna().sum()), 'PASS'],
    ['RootMoist percentile missing', int(pentad['RootMoist_pct'].isna().sum()), 'PASS'],
    ['Window-5 percentile missing', int(pentad['SM_0_100_pct_window5'].isna().sum()), 'PASS'],
    ['Primary events', len(events_primary), 'RESULT'],
    ['RootMoist sensitivity events', len(events_root), 'RESULT'],
    ['Window-5 sensitivity events', len(events_window), 'RESULT'],
    ['Primary right-censored events',
     int(events_primary['right_censored_flag'].sum()) if len(events_primary) else 0,
     'PASS'],
]

qc_df = pd.DataFrame(qc_rows, columns=['check', 'value', 'status'])
qc_df.to_csv(QC_OUT, index=False)

print('Saved Phase 2C outputs in:', PHASE2C_DIR)
for p in [PENTAD_OUT, PRIMARY_OUT, ROOT_OUT, WINDOW_OUT, QC_OUT]:
    print(' -', p)


In [ ]:
# Compact final QC
print('Pentad rows:', len(pentad))
print('5-day pentads:', (pentad['n_days'] == 5).sum())
print('6-day leap pentads:', (pentad['n_days'] == 6).sum())
print('Primary percentile missing:', pentad['SM_0_100_pct'].isna().sum())
print('Primary events:', len(events_primary))
print('Primary right-censored:', events_primary['right_censored_flag'].sum())
print('QC file:', QC_OUT)
print()
print('First 10 primary events:')
display(events_primary.head(10))


## Next step — Phase 2D

Do not tune the event thresholds based on the resulting event count.

Next we verify chronology, seasonal distribution and sensitivity agreement, then characterize precipitation-deficit / atmospheric-demand pathways **without changing the locked primary event labels**.
